# 🧹 10.12 Working with Strings & Dates

Welcome back! In the last lesson (**10.11 DataFrame Transformations**), we learned the core tools to shape our DataFrames: `select`, `filter`, and `withColumn`.

But what happens when the data *inside* those columns is a complete mess? 

In the real world, human beings enter data. They type phone numbers with dashes, parentheses, or spaces. They write salaries as "$100,000" instead of `100000`. They write dates as "10-Jan-2023" or "2023/01/10". 

As a Data Engineer, you cannot perform math on a string that says "$100,000". In this lesson, we will learn how to use PySpark's built-in functions to clean messy text and unlock the power of time-series data.

### 🎯 Learning Objectives
* Learn how to combine and split text using `concat()` and `split()`.
* Master cleaning messy text data using `regexp_replace()`.
* Understand why Dates must be converted from Strings using `to_date()`.
* Perform time-travel math using built-in Date Functions.

In [ ]:
# 0. Setup: Let's start our SparkSession and create a VERY messy DataFrame.
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("Strings_And_Dates").master("local[*]").getOrCreate()

# Here is our messy data!
# - 'full_name' needs to be split.
# - 'salary' is a string with symbols ($ and ,) making math impossible.
# - 'hire_date' is a string, not a real Date object.
data = [
    ("Alice Smith", "Engineering", "$105,000", "2021-06-15"),
    ("Bob Jones", "Sales", "$75,500", "2022-11-20"),
    ("Charlie Brown", "Marketing", "$82,000", "2023-01-10")
]
columns = ["full_name", "department", "salary", "hire_date"]

df = spark.createDataFrame(data, columns)
print("Raw, Messy DataFrame:")
df.show()
df.printSchema()

## 1. String Manipulation: `split()` and `concat()`

### 🪓 `split(column, delimiter)`
Often, multiple pieces of information are trapped in one string. For example, `full_name` contains both the first and last name. 
We can use `F.split()` to break the string apart based on a specific character (like a space " ").

### 🔗 `concat()` and `concat_ws()`
Conversely, sometimes we want to combine columns. 
* `F.concat(col1, col2)` glues strings together directly.
* `F.concat_ws(delimiter, col1, col2)` glues strings together *with a separator* (like a comma or space) in between them.

In [ ]:
print("--- Splitting and Concatenating Strings ---")

df_strings = df \
    .withColumn("first_name", F.split(F.col("full_name"), " ")[0]) \
    .withColumn("last_name", F.split(F.col("full_name"), " ")[1]) \
    .withColumn("system_username", F.concat_ws("_", F.col("department"), F.col("first_name")))

df_strings.select("full_name", "first_name", "last_name", "system_username").show(truncate=False)

## 2. Cleaning Text: `regexp_replace()`

Look at our `salary` column. It looks like this: `"$105,000"`. 

Because it has a `$` and a `,`, Spark thinks it is a piece of text (a String). If we try to do `salary * 1.10` to give a 10% raise, Spark will crash because you can't do math on text!

We need to strip out those symbols. We use `F.regexp_replace(column, pattern, replacement)`.

*(Note: 'regexp' stands for Regular Expressions, a powerful syntax for finding patterns in text. Here, we will tell it to look for `$` or `,` and replace them with nothing `""`)*.

<img src="../assets/Module_10/10_12_01.png" alt="A messy string going through a washing machine and coming out as clean, separated data blocks ready for math." width="75%">

In [ ]:
print("--- Cleaning the Salary Column ---")

# 1. Replace the symbols with nothing ("")
# The regex pattern '[$,]' means "Look for either a $ or a ,"
df_clean_salary = df.withColumn("clean_salary_string", F.regexp_replace(F.col("salary"), "[$,]", ""))

# 2. Even though it's clean, it is still technically a String. 
# We must 'cast' it to an Integer so we can do math on it!
df_final_salary = df_clean_salary.withColumn("salary_number", F.col("clean_salary_string").cast("integer"))

df_final_salary.select("salary", "clean_salary_string", "salary_number").show()
df_final_salary.printSchema()

## 3. The Magic of Dates: `to_date()`

Currently, our `hire_date` column is a String. To Spark, "2021-06-15" is just a collection of characters, no different than the word "Apple". 

If it is just a string, we cannot ask Spark: *"How many days has this person worked here?"* or *"What month were they hired?"*

To unlock date math, we must convert the string into a true Spark `DateType` object using `F.to_date()`. 

<img src="../assets/Module_10/10_12_02.png" alt="A calendar turning into a math equation, illustrating how converting strings to Date objects allows for time-based calculations." width="75%">

In [ ]:
print("--- Converting Strings to Dates ---")

# Spark is smart enough to recognize the standard "YYYY-MM-DD" format automatically.
df_dates = df.withColumn("real_hire_date", F.to_date(F.col("hire_date")))

df_dates.select("hire_date", "real_hire_date").show()

# Notice the difference in the Schema! String vs Date.
df_dates.printSchema()

## 4. Date Functions (Time Travel Math)

Now that Spark knows `real_hire_date` is a timeline, we can use powerful built-in date functions!

* `F.current_date()`: Returns today's date.
* `F.datediff(end, start)`: Calculates the number of days between two dates.
* `F.year()`, `F.month()`, `F.dayofmonth()`: Extracts specific parts of a date.

In [ ]:
print("--- Performing Date Math ---")

# Let's find out how many days each employee has worked here, 
# and extract their hiring year for a future report.

df_date_math = df_dates \
    .withColumn("today", F.current_date()) \
    .withColumn("days_employed", F.datediff(F.col("today"), F.col("real_hire_date"))) \
    .withColumn("hiring_year", F.year(F.col("real_hire_date")))

df_date_math.select("full_name", "real_hire_date", "today", "days_employed", "hiring_year").show()

## 🧑‍💻 The Modern Data Ecosystem: Role Comparison

Data cleaning is where the majority of Data Engineering time is spent. Let's see how roles view this task.

| Feature | 🏛️ Traditional DBA | 🛠️ Data Engineer (You) | 🧠 Data Scientist |
| :--- | :--- | :--- | :--- |
| **Data Cleaning** | Uses SQL `SUBSTRING` and `REPLACE`. | **Uses PySpark `split`, `regexp_replace`, and casting to build massive automated cleaning pipelines.** | Expects data to already be clean. If not, they use Pandas to clean it locally. |
| **Handling Dates** | Enforces `DATE` columns on table creation. | **Parses 50 different weird string date formats coming from messy APIs and standardizes them using `to_date()`.** | Uses dates for time-series forecasting algorithms. |
| **Time Spent** | 20% | **80% (Cleaning, formatting, and casting data is the core of ETL).** | 60% (If the DE didn't do their job well!). |

---

### 📈 Career Progression Roadmap

1. **Junior DE:** Relies heavily on basic `.replace()` and struggles when string formats change slightly (e.g., getting tripped up if a salary has a comma but no dollar sign).
2. **Mid-Level DE:** Masters **Regular Expressions (Regex)** via `regexp_replace`. Regex is a superpower that allows a Data Engineer to find complex patterns (like validating email addresses or removing all non-numeric characters) instantly.
3. **Senior DE:** Builds automated Data Quality frameworks. A Senior DE writes code that automatically detects if a date string violates the expected format, quarantines the bad row into a "Dead Letter Queue," and alerts the team without crashing the pipeline.

### 🛠️ Where Data Engineering Fits in Modern Organizations
Machine Learning models are essentially just giant math equations. You cannot feed the string `"$105,000"` into a math equation. The Data Engineer's string cleaning and type-casting pipelines are what make AI and Analytics physically possible.

## 🔑 Key Takeaways

- **`split()`:** Breaks a single string into an array of strings based on a delimiter.
- **`concat()` & `concat_ws()`:** Glues multiple columns together into a single string.
- **`regexp_replace()`:** The ultimate cleaning tool. Uses regex patterns to find and remove/replace specific characters (like symbols in currency).
- **Casting:** Even if text *looks* like a number, you must use `.cast("integer")` or `.cast("double")` to do math on it.
- **`to_date()`:** Converts string representations of dates into actual Spark Date objects, unlocking powerful time-travel functions like `datediff()` and `year()`.

## 📝 Knowledge Check Quiz

**Question 1:** You have a column called `phone` with data like `"(555)-123-4567"`. You want to remove the parentheses and dashes so it just says `"5551234567"`. Which PySpark function is best suited for this complex string replacement?
A) `F.split()`
B) `F.regexp_replace()`
C) `F.concat()`
D) `F.to_date()`

**Question 2:** Why must a Data Engineer convert a column containing dates (like "2023-12-01") from a `StringType` to a `DateType`?
A) To save hard drive space, because strings are too heavy.
B) To allow Spark to treat the data as a timeline, enabling mathematical date functions like calculating the days between two dates (`datediff`).
C) Because Spark DataFrames cannot store String types.
D) To encrypt the data.

**Question 3:** What is the difference between `concat` and `concat_ws`?
A) `concat` is for numbers, `concat_ws` is for text.
B) `concat` combines strings directly (e.g., "JohnDoe"), while `concat_ws` combines strings *with a separator* between them (e.g., "John-Doe").
C) `concat_ws` can only be used on Dates.
D) There is no difference; they are aliases for the same function.

---

*Answers: 1: B, 2: B, 3: B*

In [ ]:
# Clean up our session
spark.stop()
print("Spark session closed.")

### 🚀 What's Next?
You've mastered manipulating standard rows and columns. 

But modern Big Data isn't always perfectly flat. Often, you will receive complex JSON documents where a single column might contain an entire list of objects! 

In the next lesson, **10.13 Complex Data Types & Nested Data**, we will learn how to unpack Arrays, Structs, and Nested JSON using explosive functions. Let's go!